# Pytorch Transformation Pipeline
Will take in the ouput from Cleaning_Data.ipnyb and change into pytorch input as well as run some sanity checks. 

In [1]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import netCDF4
from pathlib import Path
from datetime import datetime
import json
import torch
import random

In [2]:
from helpers.data_funcs import make_index #need to restart kernel to get these updated!!
from helpers.data_funcs import IcepackDataset
from helpers.data_funcs import ice_days

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


## Config

In [4]:
CLEAN_NAME = "ksno16_Tsno" 
SPLIT_NAME = "Tsno"

In [5]:

window = 365 # in days, length of input series

ice_threshold = 0.15 #0.15 from Ridel talk
min_ice_obs = 14 #set to 0 if you don't want a filter on the training data, same units as window!

num_val_years = 8
training_vars = ["Tair", "fcondtop", 
                'aicen_ncat1', 'aicen_ncat2', 'aicen_ncat3', 'aicen_ncat4', 'aicen_ncat5', 
                'vicen_ncat1', 'vicen_ncat2', 'vicen_ncat3', 'vicen_ncat4', 'vicen_ncat5', 
                'vsnon_ncat1', 'vsnon_ncat2', 'vsnon_ncat3', 'vsnon_ncat4','vsnon_ncat5',
                'Tsno10_ncat1',   'Tsno10_ncat2',   'Tsno10_ncat3',   'Tsno10_ncat4',   'Tsno10_ncat5',
                'Tsno11_ncat1',   'Tsno11_ncat2',   'Tsno11_ncat3',   'Tsno11_ncat4', 'Tsno11_ncat5',
                'Tsno12_ncat1', 'Tsno12_ncat2',  'Tsno12_ncat3',  'Tsno12_ncat4',  'Tsno12_ncat5']
                #'Tsfcn1_ncat1','Tsfcn1_ncat2','Tsfcn1_ncat3','Tsfcn1_ncat4','Tsfcn1_ncat5'] 
                 #'qsno10_ncat1','qsno10_ncat2','qsno10_ncat3','qsno10_ncat4','qsno10_ncat5',
                 #'qsno11_ncat1', 'qsno11_ncat2','qsno11_ncat3','qsno11_ncat4','qsno11_ncat5',
                 #'qsno12_ncat1', 'qsno12_ncat2', 'qsno12_ncat3', 'qsno12_ncat4', 'qsno12_ncat5' ]
SEED = 42
split_var = "set" #set or loc
TEST_VALUES = [] # choosing parameter set 3 and 5 (across all locations!!), never take endpoints
VAL_VALUES = [3,8,12] # "practice test" (Be aware of the run size!)
filter_validation = True # set to false if you want to keep "ice free" obs in the validation data

In [6]:
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)
IN_DIR = Path(f"/projects/dagr5998/cleaned_data/01icepack_array_output/{CLEAN_NAME}")
SPLIT_DIR = Path(f"/projects/dagr5998/cleaned_data/02icepack_split_output/{CLEAN_NAME}/{SPLIT_NAME}")

#### Loading and Checking Data

In [7]:
X = np.load(IN_DIR / "X.npy") # input tensor
Y = np.load(IN_DIR / "Y.npy") # output tensor
sets = np.load(IN_DIR / "sets.npy") # change parameter set index
parameterizations = np.load(IN_DIR / "parameterizations.npy") # changed parameterizations
locations = np.load(IN_DIR/"locations.npy") 
sample_len = X.shape[1]

In [8]:
with open(IN_DIR / 'config.json') as f:
    config = json.load(f)
channels = config["channels"]

In [9]:
missing = set(training_vars)-set(channels)
print(missing)

set()


In [10]:
assert list(config['shape'])==list(X.shape) 
assert(X.shape[0] == Y.shape[0] == sets.shape[0] ==parameterizations.shape[0] == locations.shape[0]) #confirms the datasets match
assert set(training_vars).issubset(set(channels))
input_window  = window * (24//config["aggregated hours"])

In [11]:
var_ind_list = [channels.index(v) for v in training_vars] #select relevant vars
Xt = torch.from_numpy(X[:, :, var_ind_list].copy())
Yt = torch.from_numpy(Y)

#### Splitting Testing and Training Data

In [12]:
split_col = {"set": sets, "location": locations}[split_var]
test_mask = np.isin(split_col, TEST_VALUES) #lets the split be set at top
val_mask  = np.isin(split_col, VAL_VALUES)

train_ind = np.flatnonzero(~test_mask & ~val_mask)
val_ind   = np.flatnonzero(val_mask)
test_ind  = np.flatnonzero(test_mask)

train_index2 = np.array(make_index(train_ind, sample_len, input_window))
test_index  = np.array(make_index(test_ind,  sample_len, input_window))#[run,start timestep]
val_index = np.array(make_index(val_ind, sample_len, input_window))

first = config["first_year"]
steps_per_year =  365*(24 // config["aggregated hours"]) 

start = first + train_index2[:,1] // steps_per_year
end = first + (train_index2[:,1]+input_window-1) // steps_per_year


sample = rng.choice(np.unique(start), num_val_years, replace=False)
val_years = np.sort(sample)

year_val  = np.isin(start, val_years) & (start == end)
year_adj = np.isin(start,val_years) | np.isin(end, val_years) #if the window is something weird then the sample can be in multiple years
val_year_index = train_index2[year_val]
train_index = train_index2[~year_adj]
print(val_years)

[1978 1979 1994 2003 2008 2009 2015 2020]


In [13]:
ice_train = ice_days( X, channels, train_index, input_window, ice_threshold)
keep_train_mask = ice_train >= min_ice_obs

n_drop = np.sum(~keep_train_mask)
print(f"dropping {n_drop} of {len(keep_train_mask)} training windows due to lack of ice")

train_index = train_index[keep_train_mask]

if filter_validation:
    ice_val = ice_days( X, channels, val_index, input_window, ice_threshold)
    keep_val_mask = ice_val >= min_ice_obs

    n_drop = np.sum(~keep_val_mask)
    print(f"dropping {n_drop} of {len(keep_val_mask)} validation windows due to lack of ice")

    val_index = val_index[keep_val_mask]
    
    ice_val_year = ice_days( X, channels, val_year_index, input_window, ice_threshold)
    keep_val_year_mask = ice_val_year >= min_ice_obs

    n_drop = np.sum(~keep_val_year_mask)
    print(f"dropping {n_drop} of {len(keep_val_year_mask)} validation windows due to lack of ice")

    val_year_index = val_year_index[keep_val_year_mask]



dropping 260 of 2132 training windows due to lack of ice
dropping 66 of 588 validation windows due to lack of ice
dropping 26 of 416 validation windows due to lack of ice


In [14]:
train_runs = np.unique([r for r, b in train_index])
test_runs = np.unique([r for r, b in test_index])

### Making Dataset Class

In [15]:
Xtrain = torch.cat([Xt[r,s:s+input_window] for r,s in train_index])

In [16]:
mu = Xtrain.mean(0)#calculate the mean/sd of the remaining channel
sd = Xtrain.std(0)
print(sd.min(), sd.max())
print(training_vars[sd.argmin()], training_vars[sd.argmax()]) # highest and lowest variance channels
#plt.plot(Xt[0,:,sd.argmax()])

tensor(0.0186) tensor(18.3013)
vsnon_ncat1 fcondtop


In [17]:
val_ds = IcepackDataset(Xt, Yt, val_index, input_window,mu,sd)
val_year_ds = IcepackDataset(Xt, Yt, val_year_index, input_window,mu,sd)
train_ds = IcepackDataset(Xt, Yt, train_index, input_window,mu,sd)
test_ds  = IcepackDataset(Xt, Yt, test_index,  input_window,mu,sd)
print(len(train_ds), len(test_ds), len(val_ds), len(val_year_ds))

1872 0 522 390


In [18]:
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=8, shuffle=True) #shuffle the training data 
test_loader  = torch.utils.data.DataLoader(test_ds,  batch_size=8, shuffle=False)

x,y = next(iter(train_loader))
print(x.mean(dim=(0,1))[:5])
print(x.std(dim=(0,1))[:5])
x.shape

tensor([ 0.1304, -0.0026,  0.2021,  0.1942,  0.0030])
tensor([0.9495, 0.9742, 1.1573, 1.0208, 1.0909])


torch.Size([8, 365, 32])

#### Saved Output

In [19]:
SPLIT_DIR.mkdir(parents=True)
print(SPLIT_DIR)

/projects/dagr5998/cleaned_data/02icepack_split_output/ksno16_Tsno/Tsno


In [20]:
np.save(SPLIT_DIR / "train_index.npy", np.array(train_index, dtype=np.int32))  # (n_train, 2) [run, start]
np.save(SPLIT_DIR / "test_index.npy",  np.array(test_index,  dtype=np.int32))

np.save(SPLIT_DIR / "mu.npy",          mu.numpy())                             # (C_used,)
np.save(SPLIT_DIR / "sd.npy",          sd.numpy())

np.save(SPLIT_DIR / "ice_days_val.npy", ice_days(X, channels, val_index, input_window, ice_threshold))
np.save(SPLIT_DIR / "ice_days_val_year.npy", ice_days(X, channels, val_year_index, input_window, ice_threshold))


np.save(SPLIT_DIR / "val_index.npy", np.array(val_index, dtype=np.int32)) 
np.save(SPLIT_DIR / "val_year_index.npy", np.array(val_year_index, dtype=np.int32)) 

split_config = {
    "in_path": str(IN_DIR),
    "training_vars": training_vars,
    "var_ind_list": var_ind_list,
    "input_window_days": window,
    "input_window_timesteps": input_window,
    "split_var": split_var,
    "test_values": TEST_VALUES,
    "seed": SEED,
    "val_values":VAL_VALUES,
    "val_years":val_years.tolist(),
    "ice_threshold":ice_threshold,
    "min_ice_obs": min_ice_obs,
    "first_year":first,
    "n_val_year":len(val_year_index),
    "n_train": len(train_index),
    "n_test": len(test_index),
    "n_val": len(val_index),
    "val_filtered": filter_validation, 
    "created": datetime.now().isoformat(timespec="seconds"),
}
(SPLIT_DIR / "split_config.json").write_text(json.dumps(split_config, indent=2))

1480